# 01 - Schema
**Module:** `src/core/schema.py`
## What is it?
This module defines what a Bantrly lesson looks like...every other module imports and validates against these modules.
Uses Pydantic --  why? Pydantic catches all the wrong type, invalid args, missing fields etc., before it is fed to the LLM.

## What does this notebook explore?
1. The enum types -- valid values for grade band, domain etc
2. Building valid Lesson object from scratch
3. What happens when we pass invalid inputs
4. Inspecting the lesson as a dict ( ready for JSON serialization)


In [1]:
import sys
sys.path.insert(0, '..')

from src.core.schema import (
    GradeBand, ELADomain, LessonType, VoiceMarker,
    GuardrailStatus, PromptType,
    PracticePrompt, Hook, ModelStage, Reflect,
    LessonFlow, LessonMetadata, GuardrailCheck, GuardrailFlags,
    Lesson
)

## 1. Exploring the Enums
Enums define the only valid values for categorical fields.

In [3]:
# Print all valid values for each enum
print("GradeBand values:")
for v in GradeBand:
    print(f"  {v.name:8} → '{v.value}'")

print("\nELADomain values:")
for v in ELADomain:
    print(f"  {v.name:25} → '{v.value}'")

print("\nLessonType values:")
for v in LessonType:
    print(f"  {v.name:15} → '{v.value}'")

print("\nVoiceMarker values:")
for v in VoiceMarker:
    print(f"  {v.name:15} → '{v.value}'")

GradeBand values:
  K2       → 'K-2'
  G35      → '3-5'
  G68      → '6-8'
  G912     → '9-12'

ELADomain values:
  SPEAKING                  → 'Speaking'
  LISTENING                 → 'Listening'
  READING                   → 'Reading'
  WRITING                   → 'Writing'
  READING_TO_SPEAKING       → 'Reading → Speaking'

LessonType values:
  STORY_RETELL    → 'Story Retell'
  MISSION_BRIEF   → 'Mission Brief'
  DEBATE_DROP     → 'Debate Drop'
  TEXT_EXPLORER   → 'Text Explorer'
  LISTEN_JUDGE    → 'Listen & Judge'

VoiceMarker values:
  PRONUNCIATION   → 'Pronunciation & Articulation'
  PROSODY         → 'Prosody'
  SPEAKING_RATE   → 'Speaking Rate'
  FLUENCY         → 'Fluency & Fillers'
  VOLUME          → 'Volume Control'
  TASK_ADHERENCE  → 'Task Adherence'


## 2. Building a valid Lesson Object
Construct a complete Lesson object manually to understand what each field requires.


In [4]:
lesson = Lesson(
    lesson_id="L-K2-SPK-001",
    metadata=LessonMetadata(
        grade_band=GradeBand.K2,
        ela_domain=ELADomain.SPEAKING,
        lesson_type=LessonType.STORY_RETELL,
        theme="Nature & Animals",
        primary_skill="Retell the key events of a story in sequence",
        voice_markers=[VoiceMarker.SPEAKING_RATE, VoiceMarker.FLUENCY],
        estimated_duration_minutes=6,
        ccss_anchor="CCSS.ELA-Literacy.SL.K.4"
    ),
    lesson_flow=LessonFlow(
        hook=Hook(
            duration_seconds=70,
            content="Deep in a forest, a small turtle named Pip found a golden acorn..."
        ),
        model=ModelStage(
            duration_seconds=80,
            content="Listen to how I retell using: First... Then... Finally...",
            skill_named_explicitly="Today we are practicing: retelling a story in sequence."
        ),
        practice=[
            PracticePrompt(
                prompt_id="P1",
                type=PromptType.SUPPORTED,
                text="Tell me what happened at the beginning of Pip's story.",
                scaffold="Start with: 'First, Pip...'"
            ),
            PracticePrompt(
                prompt_id="P2",
                type=PromptType.INDEPENDENT,
                text="Now tell me the whole story — beginning, middle, and end.",
                scaffold=None
            ),
        ],
        reflect=Reflect(
            duration_seconds=45,
            voice_marker_focus=VoiceMarker.SPEAKING_RATE,
            positive_signal="A strong retell sounds steady and clear.",
            growth_signal="Try pausing after each sequence word."
        )
    ),
    guardrail_flags=GuardrailFlags(
        cognitive_load_check=GuardrailCheck(
            status=GuardrailStatus.PASS,
            message="One skill, familiar setting"
        ),
        vocabulary_ceiling_check=GuardrailCheck(
            status=GuardrailStatus.PASS,
            message="Under 30 words per prompt"
        ),
        cultural_bias_check=GuardrailCheck(
            status=GuardrailStatus.PASS,
            message="Culturally neutral animal setting"
        ),
        single_skill_check=GuardrailCheck(
            status=GuardrailStatus.PASS,
            message="Single skill confirmed"
        )
    )
)

print(lesson.summary())

[L-K2-SPK-001] GradeBand.K2 | ELADomain.SPEAKING | LessonType.STORY_RETELL | Theme: Nature & Animals | Skill: Retell the key events of a story in sequence


# 3. Testing invalid input
This is why I am using Pydantic. It catches bad data immediately.

In [7]:
from pydantic import ValidationError as PydanticValidationError

print("Attempting to create LessonMetadata with invalid grade band...\n")

try:
    bad_metadata = LessonMetadata(
        grade_band="kindergarten",   # ← invalid
        ela_domain=ELADomain.SPEAKING,
        lesson_type=LessonType.STORY_RETELL,
        theme="Animals",
        primary_skill="Retell a story",
        voice_markers=[VoiceMarker.FLUENCY],
        estimated_duration_minutes=5,
        ccss_anchor="CCSS.ELA-Literacy.SL.K.4"
    )
except PydanticValidationError as e:
    print("Pydantic caught the error!")
    print(f"Error location : {e.errors()[0]['loc']}")
    print(f"Error message  : {e.errors()[0]['msg']}")
    print(f"Invalid value  : {e.errors()[0]['input']}")

Attempting to create LessonMetadata with invalid grade band...

Pydantic caught the error!
Error location : ('grade_band',)
Error message  : Input should be 'K-2', '3-5', '6-8' or '9-12'
Invalid value  : kindergarten


In [8]:
# Try a duration that's out of range (must be 4-10 minutes)
print("Attempting lesson with duration = 25 minutes...\n")

try:
    bad_metadata = LessonMetadata(
        grade_band=GradeBand.K2,
        ela_domain=ELADomain.SPEAKING,
        lesson_type=LessonType.STORY_RETELL,
        theme="Animals",
        primary_skill="Retell a story",
        voice_markers=[VoiceMarker.FLUENCY],
        estimated_duration_minutes=25,   # ← out of range
        ccss_anchor="CCSS.ELA-Literacy.SL.K.4"
    )
except PydanticValidationError as e:
    print("Pydantic caught the error!")
    print(f"Error: {e.errors()[0]['msg']}")

Attempting lesson with duration = 25 minutes...

Pydantic caught the error!
Error: Input should be less than or equal to 10


## 4. Inspecting the lesson
The generator saves lessons as JSON.

In [10]:
import json

# Convert lesson to dict
lesson_dict = lesson.to_dict()

# Pretty print as JSON
print(json.dumps(lesson_dict, indent=2))

{
  "lesson_id": "L-K2-SPK-001",
  "metadata": {
    "grade_band": "K-2",
    "ela_domain": "Speaking",
    "lesson_type": "Story Retell",
    "theme": "Nature & Animals",
    "primary_skill": "Retell the key events of a story in sequence",
    "voice_markers": [
      "Speaking Rate",
      "Fluency & Fillers"
    ],
    "estimated_duration_minutes": 6,
    "ccss_anchor": "CCSS.ELA-Literacy.SL.K.4",
    "design_notes": null
  },
  "lesson_flow": {
    "hook": {
      "duration_seconds": 70,
      "content": "Deep in a forest, a small turtle named Pip found a golden acorn..."
    },
    "model": {
      "duration_seconds": 80,
      "content": "Listen to how I retell using: First... Then... Finally...",
      "skill_named_explicitly": "Today we are practicing: retelling a story in sequence."
    },
    "practice": [
      {
        "prompt_id": "P1",
        "type": "supported",
        "text": "Tell me what happened at the beginning of Pip's story.",
        "scaffold": "Start with: '

## Summary

| What we explored | Key takeaway |
|---|---|
| Enums | Only valid values are accepted -- no typos or invalid strings can slip through |
| Valid lesson object | All 15 classes compose cleanly into a single `Lesson` object |
| Invalid grade band | Pydantic raises immediately with a precise, readable error |
| Invalid duration | Range validation works automatically |
| Serialisation | `lesson.to_dict()` produces clean JSON-ready output |

**Design decision this validates:**
Using Pydantic as our schema layer means the LLM's output is validated
against a real data model -- not just checked with ad-hoc if-statements.
This is grounded in defensive programming principles and directly reduces
the risk of malformed lessons reaching students.